# Khipus.ai — Estadística Aplicada con Python
## Módulo 2 · Clase 3 — De la correlación a la predicción

**Docente:** Walter J. Méndez · UTEPSA / Khipus.ai
**Nombre:** _______________________________________

---

## La pregunta de hoy

> ## ¿Cuánto va a ganar un programador con 5 años de experiencia?

Es una pregunta que se puede responder **mal en una línea** y **bien en una hora**.

Hoy vamos a responderla de las dos maneras, y la diferencia entre las dos **es la clase**.

---

**Advertencia honesta antes de empezar:** al final de esta clase vas a tener un modelo que
predice sueldos, con todos los números en su lugar, y vas a descubrir que **no le podés creer**.
Eso no es un fracaso del ejercicio. Es la lección más útil que te vas a llevar del módulo.

### Cómo usar este notebook

| Marca | Qué significa |
|---|---|
| 👨‍🏫 | Lo vemos juntos. La celda ya viene resuelta. |
| 🤔 | **Predecí antes de ejecutar** y escribí tu respuesta en el chat. |
| ✍️ | `TU TURNO`: lo completás vos. Hay pista. |
| ⭐ | Desafío opcional, para quien va rápido. |
| ⚠️ | Ojo con esto. |
| 💡 | Idea para memorizar. |
| ⏱️ | Minutos previstos para ese bloque. |

📌 Esta es la versión **completa**, de profundización (120 min). La versión `Mini` es la
que dictamos en vivo en 60 minutos.

👨‍🏫 **Los bloques de hoy**

| Bloque | Qué hacemos | Qué te llevás |
|---|---|---|
| **A** | Ajustamos una recta a los datos | Predecir un número |
| **B** | Medimos cuánto se equivoca | **Decidir si le creés** |
| **C** | Le agregamos más variables | Por qué "más datos" a veces empeora |
| **D** | Probamos hasta dónde le creemos | Extrapolación y datos sucios |
| **E** | Cambiamos la pregunta a sí/no | La otra máquina: regresión logística |

💡 El sábado pasado le preguntaste a una **diferencia** si era real. Hoy le vas a hacer
exactamente la misma pregunta a una **pendiente**.

In [ ]:
import os                                        # para ver si el CSV está al lado
import numpy as np                               # operaciones numéricas
import pandas as pd                              # tablas de datos (DataFrame)
import seaborn as sns                            # gráficos estadísticos
import matplotlib.pyplot as plt                  # motor de gráficos base
import statsmodels.formula.api as smf            # 🆕 modelos con fórmulas: 'y ~ x'

sns.set_style("whitegrid")                       # mismo estilo visual de siempre

# El MISMO archivo del sábado pasado. Si no está al lado, se baja del repo de Khipus.
_LOCAL = "../datos/data_dev.csv"
_URL = ("https://raw.githubusercontent.com/Khipus-ai/Applied_Statistics_Python/"
        "main/3%20Inferential%20Statistics/data_dev.csv")

df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)

# Renombramos SOLO las cuatro columnas que vamos a usar hoy, para escribir menos.
df = df.rename(columns={
    "converted_comp_yearly": "salario",       # sueldo anual en dólares
    "years_code_pro":        "experiencia",   # años programando PROFESIONALMENTE
    "work_exp":              "exp_laboral",   # años de experiencia laboral total
    "years_code":            "anios_codigo",  # años programando (incluye aprender solo)
})

print(f"Encuestas cargadas: {df.shape[0]}")
print(f"Origen: {'carpeta local' if os.path.exists(_LOCAL) else 'repositorio de Khipus'}")

👨‍🏫 **Este archivo ya lo conocés.** Es la encuesta de desarrolladores del sábado pasado —
la misma con la que probamos si los que *ya usan* IA ganan distinto que los que *planean* usarla.

La diferencia es qué le preguntamos. La semana pasada: *"¿esta diferencia entre dos grupos es
real?"*. Hoy: *"¿puedo **predecir** el sueldo de alguien que no está en la tabla?"*.

In [ ]:
# Las cuatro columnas de hoy
df[["salario", "experiencia", "exp_laboral", "anios_codigo"]].head()

In [ ]:
df["salario"].describe().round(0)

⚠️ **Antes de modelar nada, mirá los extremos.**

El salario **mínimo** de la tabla es de **3 dólares al año**. El **máximo**, 1.200.000.

Los dos son números válidos: Python los suma, los promedia y no se queja. Pero un sueldo anual
de 3 dólares no existe — es alguien que respondió cualquier cosa, o que puso su sueldo por hora,
o que se equivocó de casilla.

💡 **Ningún programa te va a avisar de esto.** El único filtro que existe es que vos sepas de
qué estás hablando. Lo dejamos adentro a propósito: quiero que veas, más adelante, cuánto daño
hace.

---
# Bloque A · Del montón de puntos a una recta

⏱️ *32 minutos*

🤔 **Predicción — respondé en el chat antes de correr la celda:**

Vamos a dibujar cada encuestado como un punto: años de experiencia en el eje horizontal,
sueldo en el vertical.

**¿Qué vas a ver?**
- **(A)** Una nube que sube claramente: más experiencia, más sueldo.
- **(B)** Una nube que sube, pero muy desordenada.
- **(C)** Una mancha sin forma.

Escribí **A**, **B** o **C**. No hay trampa, pero tampoco es obvio.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df["experiencia"], df["salario"], alpha=0.35, color="#4C9BE8", edgecolor="none")

plt.title("¿El sueldo sube con la experiencia?", fontsize=14, fontweight="bold")
plt.xlabel("Años programando profesionalmente")
plt.ylabel("Sueldo anual (USD)")
plt.show()

👨‍🏫 **La respuesta honesta es (B).**

Se ve una tendencia hacia arriba, sí. Pero mirá la dispersión vertical: para *cualquier* valor de
experiencia hay gente ganando 20 mil y gente ganando 200 mil.

Ahora viene la idea central de la clase: **trazar la única recta que mejor pasa por el medio de
esa nube**. Esa recta es el modelo.

Una recta se describe con dos números:

$$\text{salario} = \beta_0 + \beta_1 \times \text{experiencia}$$

- **β₀ (intercepto)** — dónde arranca la recta: el sueldo predicho para 0 años de experiencia.
- **β₁ (pendiente o coeficiente)** — cuánto sube el sueldo por **cada año más** de experiencia.

Python calcula los dos por nosotros.

In [ ]:
# 'salario ~ experiencia' se lee: "explicá el salario en función de la experiencia".
# Es la misma notación que usa la estadística en papel, y la vamos a reusar toda la clase.
modelo = smf.ols("salario ~ experiencia", data=df).fit()

modelo.summary()

👨‍🏫 **Esa tabla intimida, así que vamos a leerla por partes.** Hoy nos importan cuatro números:
**tres viven en la tabla de abajo y uno arriba, en la cabecera.**

| Dónde mirar | Qué es | Para qué sirve |
|---|---|---|
| `Intercept` → `coef` | **β₀** | el arranque de la recta |
| `experiencia` → `coef` | **β₁** | cuánto sube por año |
| `experiencia` → `P>\|t\|` | **el p-valor** 👈 el del sábado pasado | ¿la pendiente es real o es ruido? |
| `R-squared` | **R²** | lo vemos en el Bloque B |

Los sacamos en limpio:

In [ ]:
b0 = modelo.params["Intercept"]
b1 = modelo.params["experiencia"]
p_valor = modelo.pvalues["experiencia"]

print(f"β₀ (intercepto) = ${b0:,.0f}")
print(f"β₁ (pendiente)  = ${b1:,.0f} por cada año de experiencia")
print(f"p-valor de la pendiente = {p_valor:.2e}")

💡 **El puente con la clase pasada.**

Ese p-valor de `2.32e-24` es un 0,000...0232 con 23 ceros adelante. Con el mismo criterio que
usaste el sábado pasado (α = 0,05), **rechazamos la hipótesis nula** de que la pendiente sea cero.

Traducido: *la relación entre experiencia y sueldo **no** es casualidad*. Existe de verdad.

⚠️ Guardate esa frase. En el Bloque B vamos a ver que **"existe de verdad" y "me sirve para algo"
son dos cosas completamente distintas**, y confundirlas es el error más caro de la estadística
aplicada.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df["experiencia"], df["salario"], alpha=0.30, color="#4C9BE8", edgecolor="none",
            label="Cada punto = un encuestado")

# La recta del modelo, dibujada sobre el rango real de experiencia
x_linea = np.linspace(df["experiencia"].min(), df["experiencia"].max(), 100)
plt.plot(x_linea, b0 + b1 * x_linea, color="red", linewidth=2.5,
         label=f"Modelo: ${b0:,.0f} + ${b1:,.0f} × años")

plt.title("La recta que mejor pasa por el medio de la nube", fontsize=14, fontweight="bold")
plt.xlabel("Años programando profesionalmente")
plt.ylabel("Sueldo anual (USD)")
plt.legend()
plt.show()

### ✍️ TU TURNO 1

Ya tenés `b0` y `b1`. Usá la fórmula de la recta —a mano, sin funciones nuevas— para calcular el
sueldo que el modelo predice para alguien con **5 años** de experiencia profesional.

💡 *Pista:* es una multiplicación y una suma: `b0 + b1 * anios`.

In [ ]:
# TU TURNO 1: predecir el sueldo para 5 años de experiencia, con la fórmula de la recta
# Tu código aquí

In [ ]:
# (celda de apoyo — el número que acabás de calcular, para poder seguir usándolo)
prediccion = b0 + b1 * 5
print(f"Predicción del modelo para 5 años: ${prediccion:,.0f}")

---
### 👨‍🏫 Un paso atrás: la correlación

Antes de trazar rectas conviene medir **cuánto se acompañan** dos variables. Eso es la
**correlación**, un número entre −1 y 1:

| Valor | Qué significa |
|---|---|
| **+1** | cuando una sube, la otra sube exactamente en proporción |
| **0** | no se acompañan de ninguna manera lineal |
| **−1** | cuando una sube, la otra baja en proporción |

⚠️ La correlación **no distingue causa de efecto**, y solo detecta relaciones **rectas**: dos
variables con una relación en forma de U pueden dar correlación 0 y estar perfectísimamente
relacionadas.

In [ ]:
columnas = ["salario", "experiencia", "exp_laboral", "anios_codigo"]
correlaciones = df[columnas].corr()

print(correlaciones.round(3).to_string())

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Mapa de calor de correlaciones", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

👨‍🏫 **Leé la primera columna:** la correlación entre `salario` y `experiencia` es de apenas
**0,29**. Positiva —así que la nube sube— pero floja.

💡 Guardate ese 0,29. En el Bloque B vas a ver que **el R² de la regresión simple es exactamente
ese número al cuadrado** (0,29² ≈ 0,084). No es coincidencia: en una regresión de una sola
variable, R² *es* la correlación al cuadrado. Por eso la letra.

Y mirá el resto de la tabla: `experiencia`, `exp_laboral` y `anios_codigo` se correlacionan entre
sí muchísimo más (0,80–0,89) de lo que cualquiera se correlaciona con el salario. **Eso va a
explotar en el Bloque C.**

### 👨‍🏫 Cómo se elige "la mejor" recta

Por una nube de puntos pasan infinitas rectas. ¿Por qué Python elige *esa*?

El criterio se llama **mínimos cuadrados** (*ordinary least squares*, de ahí el `ols` que
escribimos): entre todas las rectas posibles, elige **la que hace más chica la suma de los
residuos al cuadrado**.

¿Por qué al cuadrado y no el residuo a secas? Por dos motivos:

1. Si no los elevás al cuadrado, los errores positivos y negativos **se cancelan** y una recta
   pésima puede dar error total cero.
2. Elevar al cuadrado **castiga más los errores grandes**, que suelen ser los que más importan.

📌 El cómo lo encuentra —derivadas y descenso del gradiente— es materia del Módulo 4. Por ahora
alcanza con saber **qué** está optimizando.

### ✍️ TU TURNO 2

Interpretá el coeficiente **en palabras**, que es lo que te van a pedir en el examen.

Completá la frase con el valor de `b1` y respondé: si dos personas se llevan **10 años** de
experiencia, ¿cuánta diferencia de sueldo predice el modelo entre ellas?

💡 *Pista:* `b1 * 10`.

In [ ]:
# TU TURNO 2: la diferencia que predice el modelo entre dos personas con 10 años de diferencia
# Tu código aquí

## Primera respuesta a la pregunta de hoy

> **¿Cuánto gana un programador con 5 años de experiencia?**
>
> ## $80.853 al año.

Suena razonable. Es un número concreto, salió de 1.183 datos reales, y la pendiente que lo produjo
tiene un p-valor de 10⁻²⁴.

**Si la clase terminara acá, te irías con una respuesta equivocada.** No porque el número esté mal
calculado —está perfecto— sino porque todavía no sabemos **cuánto se equivoca**.

Eso es el Bloque B.

---
# Bloque B · ¿Cuánto le erramos?

⏱️ *38 minutos* · **Este es el bloque más importante de la clase.**

👨‍🏫 **El residuo.**

Para cada persona de la tabla tenemos dos números: lo que **gana de verdad** y lo que el
**modelo predice** para ella. La resta entre los dos se llama **residuo**:

$$\text{residuo} = \text{valor real} - \text{valor predicho}$$

- Residuo **positivo** → gana **más** de lo que el modelo esperaba.
- Residuo **negativo** → gana **menos**.
- Residuo **cero** → el modelo le pegó justo.

Los residuos son la parte que el modelo **no explica**. Y son, lejos, lo más informativo que tiene
una regresión.

In [ ]:
df_modelo = df.dropna(subset=["salario", "experiencia"]).copy()

df_modelo["predicho"] = modelo.predict(df_modelo)
df_modelo["residuo"] = df_modelo["salario"] - df_modelo["predicho"]

df_modelo[["experiencia", "salario", "predicho", "residuo"]].head(8).round(0)

In [ ]:
# Los primeros 60 encuestados, con su error dibujado como una línea vertical
sub = df_modelo.head(60)

plt.figure(figsize=(11, 6))
plt.scatter(sub["experiencia"], sub["salario"], color="#4C9BE8", zorder=3, label="Sueldo real")
plt.plot(x_linea, b0 + b1 * x_linea, color="red", linewidth=2, label="Predicción del modelo")

# Cada línea gris es UN residuo
for _, fila in sub.iterrows():
    plt.plot([fila["experiencia"], fila["experiencia"]],
             [fila["salario"], fila["predicho"]],
             color="gray", linewidth=0.9, alpha=0.7, zorder=2)

plt.title("Cada línea gris es un error del modelo", fontsize=14, fontweight="bold")
plt.xlabel("Años programando profesionalmente")
plt.ylabel("Sueldo anual (USD)")
plt.legend()
plt.show()

🤔 **Predicción obligatoria — esta es la pregunta del día. Contestá en el chat.**

Vamos a resumir todas esas líneas grises en **un solo número**: de cuánto es el error **típico**
del modelo.

**¿Cuánto te parece que da?**
- **(A)** unos $5.000
- **(B)** unos $20.000
- **(C)** unos $40.000
- **(D)** más de $70.000

Comprometete con una letra **antes** de correr la celda que sigue.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

real = df_modelo["salario"]
pred = df_modelo["predicho"]

mae = mean_absolute_error(real, pred)              # error absoluto promedio
mse = mean_squared_error(real, pred)               # error cuadrático promedio
rmse = np.sqrt(mse)                                # raíz → vuelve a estar en dólares

print(f"MAE  (error promedio)          = ${mae:,.0f}")
print(f"MSE  (castiga los errores grandes) = {mse:,.0f}  ← dólares al cuadrado, ilegible")
print(f"RMSE (el que se reporta)       = ${rmse:,.0f}")

👨‍🏫 **Las tres métricas, en una línea cada una:**

| Métrica | Qué es | Cuándo |
|---|---|---|
| **MAE** | el error promedio, en dólares | la más fácil de explicar a un jefe |
| **MSE** | el promedio de los errores **al cuadrado** | castiga fuerte los errores grandes; queda en dólares² |
| **RMSE** | la raíz del MSE → **vuelve a dólares** | **la que se reporta**, y la que vamos a usar |

💡 El RMSE es, en criollo, **el tamaño típico de una línea gris del gráfico anterior**.

## 💥 El momento de la clase

El RMSE del modelo es de **$78.343**.

El sueldo **promedio** de toda la tabla es de **$90.684**.

$$\frac{78.343}{90.684} = 86\%$$

> ## El modelo se equivoca, típicamente, en el 86% del sueldo.

Volvamos entonces a la respuesta del Bloque A, ahora dicha con honestidad:

In [ ]:
print(f"Respuesta ingenua:  ${prediccion:,.0f}")
print()
print(f"Respuesta honesta:  ${prediccion:,.0f}  ±  ${rmse:,.0f}")
print(f"                    o sea, entre ${prediccion - rmse:,.0f} y ${prediccion + rmse:,.0f}")

💡 **Decir "vas a ganar entre 2.500 y 159.000 dólares" es no decir nada.**

Es como quedar en encontrarse "entre las 3 de la tarde y las 11 de la noche". Técnicamente
verdadero. Absolutamente inútil.

Y ahora la parte incómoda: **la pendiente sigue teniendo un p-valor de 10⁻²⁴**. Sigue siendo
real. Las dos cosas son ciertas al mismo tiempo:

> ## 💡 Estadísticamente significativo ≠ útil

El p-valor responde *"¿existe la relación?"*. El RMSE responde *"¿me sirve para predecir?"*.
**Son preguntas distintas y pueden tener respuestas opuestas.**

In [ ]:
print(f"R² = {modelo.rsquared:.4f}")
print(f"El modelo explica el {modelo.rsquared * 100:.1f}% de la variación de los sueldos.")
print(f"El otro {100 - modelo.rsquared * 100:.1f}% se debe a cosas que el modelo no conoce.")

👨‍🏫 **El R² (erre cuadrado)** va de 0 a 1 y responde: *¿cuánta de la variación de los sueldos
logra explicar la experiencia?*

Una forma más útil de leerlo: **cuánto mejora el modelo respecto de simplemente predecirle a todo
el mundo el promedio**. Un R² de 0,084 significa que la recta apenas mejora un 8% sobre decir
"todos ganan $90.684".

⚠️ **¿Es 0,084 un fracaso? No necesariamente.** *An Introduction to Statistical Learning*
(Stanford) lo dice explícitamente: en problemas de marketing, psicología o biología —donde
influyen muchísimos factores desconocidos— *"un R² bien por debajo de 0,1 puede ser más realista"*.

El sueldo de una persona depende del país, la empresa, la carrera, la suerte, la negociación,
el idioma. Pedirle a **una sola variable** que explique todo eso era, desde el principio,
demasiado pedir.

💡 **El R² bajo no dice "hiciste mal la cuenta". Dice "el mundo es más complicado que tu modelo".**
Reportarlo con honestidad es hacer bien el trabajo.

⚠️ **La otra forma de engañarse: un R² alto que tampoco significa nada.**

En el sitio de Tyler Vigen hay una correlación real, calculada sobre datos reales:

> **Películas estrenadas de Nicolas Cage** vs. **inspectores de la TSA en Dakota del Norte**
> **r² = 0,814** · **p = 0,00014**

R² altísimo, p-valor ínfimo, y absolutamente ninguna relación entre las dos cosas.

💡 Entre este ejemplo y el nuestro tenés las **dos formas de mentirte con una regresión**:
creerle a un p-valor chico (nuestro caso) y creerle a un R² grande (el de Cage). El número nunca
alcanza: hace falta que la relación **tenga sentido**.

In [ ]:
# El cuarteto de Anscombe: cuatro conjuntos de datos con estadísticas casi idénticas
anscombe = sns.load_dataset("anscombe")

for nombre, grupo in anscombe.groupby("dataset"):
    r2 = smf.ols("y ~ x", data=grupo).fit().rsquared
    print(f"Grupo {nombre}:  media de x = {grupo['x'].mean():.1f}   "
          f"media de y = {grupo['y'].mean():.2f}   R² = {r2:.2f}")

In [ ]:
sns.lmplot(data=anscombe, x="x", y="y", col="dataset", hue="dataset",
           col_wrap=2, height=3, ci=None,
           scatter_kws={"s": 60, "alpha": 0.8})
plt.show()

💡 **Los cuatro tienen el mismo R² = 0,67 y la misma recta.** Y solo el primero es una relación
lineal de verdad: el segundo es una curva, el tercero tiene un dato mal cargado, el cuarto es
un solo punto mandando sobre todos los demás.

> ## Nunca creas un número de regresión sin haber mirado el gráfico.

---
### 👨‍🏫 Diagnóstico: mirar los residuos, no solo resumirlos

El RMSE resume todos los residuos en un número. Pero un número esconde **dónde** falla el modelo.
Para eso hay dos gráficos que se hacen siempre, en este orden.

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.hist(df_modelo["residuo"], bins=50, color="#4C9BE8", edgecolor="black")
plt.axvline(0, color="red", linestyle="--", linewidth=2, label="Residuo = 0 (predicción perfecta)")

plt.title("1) ¿Cómo se reparten los errores?", fontsize=13, fontweight="bold")
plt.xlabel("Residuo (USD)")
plt.ylabel("Cantidad de personas")
plt.legend()
plt.show()

print(f"Residuo promedio: ${df_modelo['residuo'].mean():,.2f}   <- siempre da ~0, por construcción")
print(f"Residuo mínimo:   ${df_modelo['residuo'].min():,.0f}")
print(f"Residuo máximo:   ${df_modelo['residuo'].max():,.0f}")

👨‍🏫 **Lo que buscás en este gráfico:** una campana centrada en cero. Lo que ves acá es una
campana **con una cola larguísima a la derecha**: hay gente que gana muchísimo más de lo que el
modelo predice, y casi nadie que gane muchísimo menos (no se puede ganar menos que cero).

⚠️ El residuo promedio da **cero siempre**, en cualquier regresión. No es una señal de que el
modelo esté bien: es una consecuencia matemática de cómo se elige la recta. **No lo uses como
métrica** — para eso están el MAE y el RMSE, que toman el valor absoluto o el cuadrado.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df_modelo["predicho"], df_modelo["residuo"], alpha=0.3, color="#4C9BE8",
            edgecolor="none")
plt.axhline(0, color="red", linestyle="--", linewidth=2)

plt.title("2) Residuos contra valores predichos", fontsize=13, fontweight="bold")
plt.xlabel("Sueldo que predice el modelo (USD)")
plt.ylabel("Residuo (USD)")
plt.show()

👨‍🏫 **Este es EL gráfico de diagnóstico de la regresión.** En el eje horizontal, lo que el modelo
predijo; en el vertical, cuánto le erró.

**Lo que querrías ver:** una nube sin forma, repartida parejo arriba y abajo de la línea roja.
Eso significa "el modelo se equivoca al azar, no tiene un sesgo".

**Lo que se ve acá:** una nube que se **abre hacia la derecha**. Cuanto más alto es el sueldo
predicho, más grandes son los errores. Eso tiene nombre: **heterocedasticidad** (la varianza del
error no es constante).

💡 Traducido al oficio: **el modelo es menos confiable justo donde los sueldos son más altos.**
Un solo número —el RMSE— no te podía decir eso.

### ⚠️ Los cuatro supuestos de la regresión lineal

Toda regresión asume cuatro cosas. No hace falta que las memorices hoy, pero sí que sepas que
existen, porque el capítulo del curso las menciona en §4.3 y **el examen puede preguntarlas**:

| Supuesto | Qué quiere decir | Cómo se chequea |
|---|---|---|
| **Linealidad** | la relación real *es* una recta | el scatter con la recta encima |
| **Independencia** | un caso no influye en otro | conocimiento del dominio |
| **Homocedasticidad** | el error tiene el mismo tamaño en todo el rango | residuos vs. predichos 👆 |
| **Normalidad de los residuos** | los errores se reparten en campana | el histograma de residuos 👆 |

En nuestro modelo, **dos de los cuatro están claramente violados**: los residuos no son normales
(cola a la derecha) y no son homocedásticos (la nube se abre).

💡 **Y eso no invalida lo que hicimos.** Significa que el modelo es todavía menos confiable de lo
que ya sabíamos. Reportarlo es parte del trabajo; ocultarlo, no.

### ✍️ TU TURNO 3

Ajustá un modelo nuevo usando `exp_laboral` (años de experiencia laboral total) en lugar de
`experiencia`, y mostrá su **R²**. ¿Explica más o menos que el anterior (0,0841)?

💡 *Pista:* copiá la línea de `smf.ols` del Bloque A y cambiale una sola palabra. El R² sale
con `.rsquared`.

In [ ]:
# TU TURNO 3: ajustar salario ~ exp_laboral y comparar su R² con el del modelo anterior
# Tu código aquí

---
# Bloque C · La trampa de los gemelos

⏱️ *18 minutos*

🤔 **Predicción — al chat:**

Si una variable explica poco, la reacción natural es **agregar más**. Vamos a meter las tres al
mismo tiempo: `experiencia`, `exp_laboral` y `anios_codigo`.

**¿Qué va a pasar con el p-valor de `experiencia`, que era 10⁻²⁴?**
- **(A)** Sigue siendo chiquito: la variable es buena.
- **(B)** Se hace todavía más chico.
- **(C)** Se arruina.

In [ ]:
modelo_multiple = smf.ols("salario ~ experiencia + exp_laboral + anios_codigo", data=df).fit()

comparacion = pd.DataFrame({
    "p-valor SOLA": [modelo.pvalues["experiencia"], np.nan, np.nan],
    "p-valor JUNTAS": [modelo_multiple.pvalues["experiencia"],
                       modelo_multiple.pvalues["exp_laboral"],
                       modelo_multiple.pvalues["anios_codigo"]],
}, index=["experiencia", "exp_laboral", "anios_codigo"])

print(comparacion.round(4).to_string())
print()
print(f"R² con 1 variable  = {modelo.rsquared:.4f}")
print(f"R² con 3 variables = {modelo_multiple.rsquared:.4f}")

## 💥 La variable estrella se apagó

`experiencia` pasó de un p-valor de **0,0000000000000000000000232** a **0,174**.

Con el criterio del sábado pasado (α = 0,05), **ya no es significativa**. La misma variable, los
mismos datos. Lo único que cambió es que le pusimos compañía.

¿Por qué? Miremos cuánto se parecen entre sí las tres variables:

In [ ]:
correlaciones = df[["experiencia", "exp_laboral", "anios_codigo"]].corr()
print(correlaciones.round(3).to_string())

👨‍🏫 **Correlaciones de 0,80 a 0,89 entre sí. Las tres son casi la misma variable.**

Esto se llama **colinealidad** (o multicolinealidad, cuando son varias). Cuando dos predictores
dicen casi lo mismo, el modelo **no puede repartir el crédito** entre ellos: no sabe si el sueldo
sube por la experiencia profesional o por los años programando, porque van siempre juntos.

Al no poder decidir, el modelo **infla la incertidumbre de los dos**. Y un coeficiente con mucha
incertidumbre tiene, por definición, un p-valor grande.

💡 Es como preguntarles a tres hermanos que viven juntos qué pasó anoche. Las tres respuestas son
casi idénticas: sumás testigos y no sumás ni un dato nuevo.

⚠️ **Y ojo con el R²:** subió de 0,0841 a 0,0900. Parece una mejora, pero **el R² sube siempre que
agregás variables**, aunque sean basura. Nunca uses el R² para decidir si valió la pena agregar
una variable.

⭐ **Desafío opcional — el VIF**

Existe una medida estándar de colinealidad, el **Factor de Inflación de la Varianza (VIF)**. La
regla de oficio: **VIF > 5 preocupa, VIF > 10 es problema serio**.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X = sm.add_constant(df[["experiencia", "exp_laboral", "anios_codigo"]].dropna())

for i, nombre in enumerate(X.columns):
    if nombre == "const":
        continue
    print(f"VIF de {nombre:14s} = {variance_inflation_factor(X.values, i):5.2f}")

---
### 💥 Un nulo, dos herramientas, dos comportamientos

La columna `anios_codigo` tiene **exactamente un valor faltante** entre 1.183 filas. Uno solo.

🤔 **Al chat, antes de correr:** ¿un solo dato faltante entre 1.183 puede romper algo?
**SÍ** o **NO**.

In [ ]:
print(f"Filas en la tabla:            {len(df)}")
print(f"Nulos en anios_codigo:        {df['anios_codigo'].isna().sum()}")
print(f"Filas que USÓ smf.ols:        {int(modelo_multiple.nobs)}")
print()
print("👆 statsmodels descartó la fila incompleta. Y NO te avisó de ninguna manera.")

In [ ]:
from sklearn.linear_model import LinearRegression

# La MISMA regresión, con la librería que vas a usar en tu trabajo práctico
X = df[["experiencia", "exp_laboral", "anios_codigo"]]
y = df["salario"]

try:
    LinearRegression().fit(X, y)
    print("Funcionó sin problemas 🎉")
except ValueError as error:
    print("💥 ERROR — Python no pudo hacerlo:")
    print(f"   ValueError: {error}")
    print()
    print("👆 Leé el mensaje. Te está diciendo EXACTAMENTE cuál es el problema.")

👨‍🏫 **`Input X contains NaN`** — "la entrada X contiene un valor faltante". No hay que adivinar
nada: el mensaje dice qué pasó y dónde.

Y fijate en la asimetría, que es lo importante:

| Herramienta | Qué hizo con el mismo nulo |
|---|---|
| `statsmodels` | Lo **tiró en silencio**. Tu modelo se ajustó con 1.182 filas y vos creías que eran 1.183 |
| `scikit-learn` | **Se negó a seguir** y te lo dijo |

💡 **El que grita te está haciendo un favor.** El silencioso es el peligroso: nada falla, todo
"funciona", y tu `n` cambió sin que te enteres. Por eso se revisan los nulos **antes** de modelar,
no cuando algo explota.

In [ ]:
# La forma correcta: decidir explícitamente qué hacemos con los faltantes
datos_limpios = df.dropna(subset=["salario", "experiencia", "exp_laboral", "anios_codigo"])

print(f"Antes:   {len(df)} filas")
print(f"Después: {len(datos_limpios)} filas   (descartamos {len(df) - len(datos_limpios)} a propósito)")

---
# Bloque D · Hasta dónde le podés creer

⏱️ *18 minutos — solo en la versión completa*

🤔 **Predicción — al chat:**

El modelo es una fórmula: `salario = β₀ + β₁ × años`. Nada le impide meterle cualquier número.

**¿Qué pasa si le pido el sueldo de alguien con 80 años de experiencia? ¿Y con −5?**

In [ ]:
print(f"Rango REAL de experiencia en los datos: {df['experiencia'].min():.0f} a {df['experiencia'].max():.0f} años")
print()

for anios in [-5, 0, 5, 25, 50, 80]:
    print(f"  {anios:3d} años  ->  ${b0 + b1 * anios:>10,.0f}")

👨‍🏫 **El modelo contesta siempre, incluso donde no tiene datos.** Para 80 años de experiencia
—que no existe en la tabla ni en la realidad— te devuelve una cifra con toda seriedad.

Eso se llama **extrapolación**: usar el modelo fuera del rango en el que fue entrenado. Es uno de
los errores más comunes y más caros.

⚠️ **La regla:** un modelo solo vale dentro del rango de datos con el que lo ajustaste. Fuera de
ahí no está prediciendo, **está inventando**.

💡 El ejemplo clásico de la literatura: un modelo de precios de casas entrenado con casas normales,
al que se le pide el precio de un lote vacío, devuelve **menos 522.202 dólares**. Nadie te va a
pagar por llevarte un terreno.

In [ ]:
plt.figure(figsize=(11, 5))
plt.scatter(df["experiencia"], df["salario"], alpha=0.25, color="#4C9BE8", edgecolor="none",
            label="Datos reales")

x_real = np.linspace(df["experiencia"].min(), df["experiencia"].max(), 100)
x_fuera = np.linspace(df["experiencia"].max(), 80, 100)

plt.plot(x_real, b0 + b1 * x_real, color="darkgreen", linewidth=3, label="Zona con datos")
plt.plot(x_fuera, b0 + b1 * x_fuera, color="red", linewidth=3, linestyle="--",
         label="Extrapolación: pura fe")

plt.title("El modelo contesta igual donde no sabe nada", fontsize=14, fontweight="bold")
plt.xlabel("Años programando profesionalmente")
plt.ylabel("Sueldo anual (USD)")
plt.legend()
plt.show()

### 👨‍🏫 El dev de $3, otra vez

Al principio de la clase vimos que había alguien declarando un sueldo de **3 dólares al año**.
Lo dejamos adentro a propósito. Veamos cuánto daño hizo.

In [ ]:
# Sacamos los sueldos imposibles: menos de $1.000 al año no es un sueldo
sin_absurdos = df[df["salario"] >= 1000].dropna(subset=["experiencia"])
modelo_limpio = smf.ols("salario ~ experiencia", data=sin_absurdos).fit()

rmse_limpio = np.sqrt(((sin_absurdos["salario"] - modelo_limpio.fittedvalues) ** 2).mean())

print(f"Filas descartadas: {len(df) - len(sin_absurdos)}")
print()
print(f"{'':22s} {'CON absurdos':>14s} {'SIN absurdos':>14s}")
print(f"{'β₁ (pendiente)':22s} {b1:>14,.0f} {modelo_limpio.params['experiencia']:>14,.0f}")
print(f"{'R²':22s} {modelo.rsquared:>14.4f} {modelo_limpio.rsquared:>14.4f}")
print(f"{'RMSE':22s} {rmse:>14,.0f} {rmse_limpio:>14,.0f}")

💡 **Sacar los sueldos imposibles casi no mueve la aguja.** El modelo sigue explicando poquísimo.

Eso es un hallazgo, no un fracaso del ejercicio: **el problema no eran unos pocos datos sucios.**
El problema es que la experiencia, sola, no explica el sueldo. Si el R² hubiera saltado de 0,08 a
0,60, la historia sería otra — y saberlo requería probarlo, no suponerlo.

⚠️ Y fijate en la trampa que evitamos: si hubiéramos limpiado los datos **antes** de ver el
diagnóstico, nos habríamos convencido de que "el modelo mejoró por la limpieza" sin tener con qué
compararlo.

### ✍️ TU TURNO 4

El otro extremo también es sospechoso: hay alguien declarando **$1.200.000** al año.

Ajustá un modelo descartando los sueldos por encima de **$500.000** y compará su R² y su RMSE
contra los del modelo original. ¿Cambia la conclusión de la clase?

💡 *Pista:* copiá el patrón de la celda anterior y cambiá la condición del filtro.

In [ ]:
# TU TURNO 4: modelo sin los sueldos mayores a $500.000, y comparación
# Tu código aquí

---
### 📌 El otro dialecto: `scikit-learn` (el de tu trabajo práctico)

⚠️ **Prestá atención a este bloque: es la sintaxis exacta que te pide el Assignment 3.**

Toda la clase usamos `smf.ols("salario ~ experiencia")` porque nos daba los p-valores, que eran
el corazón de hoy. Pero la misma regresión se escribe de otra manera con `scikit-learn`, que es
la librería del machine learning y la que vas a necesitar el domingo.

Es el **mismo modelo**, escrito distinto:

In [ ]:
from sklearn.linear_model import LinearRegression

x = datos_limpios["experiencia"]
y = datos_limpios["salario"]

# ⚠️ El paso que a todos nos confunde la primera vez:
# sklearn exige que las variables predictoras vengan en una tabla de 2 dimensiones
# (filas × columnas), no en una lista simple. reshape(-1, 1) hace exactamente eso:
# "acomodame esto en UNA columna, con las filas que hagan falta".
x_matriz = x.values.reshape(-1, 1)

reg = LinearRegression()          # 1. creamos el modelo
reg.fit(x_matriz, y)              # 2. lo entrenamos con los datos

print(f"Intercepto (β₀) : ${reg.intercept_:,.0f}")
print(f"Coeficiente (β₁): ${reg.coef_[0]:,.0f}")
print(f"R² (.score)     : {reg.score(x_matriz, y):.4f}")
print(f"Predicción 5 años: ${reg.predict(np.array([[5]]))[0]:,.0f}")

👨‍🏫 **Comparalo con lo del Bloque A: son los mismos cuatro números.** Dos librerías, dos
sintaxis, un solo modelo.

| Lo que querés | `statsmodels` (hoy) | `scikit-learn` (tu TP) |
|---|---|---|
| Ajustar | `smf.ols("y ~ x", data=df).fit()` | `LinearRegression().fit(x_matriz, y)` |
| Intercepto β₀ | `.params["Intercept"]` | `.intercept_` |
| Coeficiente β₁ | `.params["x"]` | `.coef_[0]` |
| R² | `.rsquared` | `.score(x_matriz, y)` |
| Predecir | `.predict(tabla)` | `.predict(np.array([[5]]))` |
| **p-valores** | `.pvalues` ✅ | **no los da** ❌ |

💡 Por eso existen las dos: **sklearn sirve para predecir, statsmodels para entender.**

---
# Bloque E · La otra máquina: cuando la respuesta es sí o no

⏱️ *14 minutos*

👨‍🏫 Hasta acá predijimos un **número** (un sueldo). Pero muchísimas preguntas de negocio no son
números, son **sí o no**:

*¿este cliente va a pagar? ¿este alumno va a aprobar? ¿esta transacción es fraude?*

Nuestra tabla tiene una columna que no tocamos el sábado pasado: `us_or_not`, si la persona vive
en Estados Unidos o no. Preguntemos: **¿se puede adivinar dónde vive alguien mirando su sueldo?**

In [ ]:
datos_log = df.dropna(subset=["salario", "us_or_not"]).copy()
datos_log["es_us"] = (datos_log["us_or_not"] == "US").astype(int)   # 1 = sí, 0 = no

print(datos_log["us_or_not"].value_counts().to_string())
print(f"\nProporción que vive en EEUU: {datos_log['es_us'].mean():.1%}")

🤔 **Al chat:** si le pedimos a una **recta** que prediga algo que solo puede valer 0 o 1,
¿qué va a salir mal?

In [ ]:
plt.figure(figsize=(11, 5))
plt.scatter(datos_log["salario"], datos_log["es_us"], alpha=0.25, color="#4C9BE8")

recta = smf.ols("es_us ~ salario", data=datos_log).fit()
x_s = np.linspace(0, datos_log["salario"].max(), 200)
plt.plot(x_s, recta.params["Intercept"] + recta.params["salario"] * x_s,
         color="red", linewidth=2.5, label="Una RECTA (mal)")

plt.axhline(1, color="gray", linestyle=":", linewidth=1)
plt.axhline(0, color="gray", linestyle=":", linewidth=1)
plt.title("Una recta no sabe que la probabilidad vive entre 0 y 1",
          fontsize=14, fontweight="bold")
plt.xlabel("Sueldo anual (USD)")
plt.ylabel("¿Vive en EEUU? (0 = no, 1 = sí)")
plt.legend()
plt.show()

👨‍🏫 **La recta se escapa.** Mirá el extremo derecho: para el sueldo más alto del archivo
($1.200.000) la recta predice **3,35**, o sea una probabilidad del **335%**. Por abajo cruza el
cero apenas (−0,0007 en `salario = 0`), pero lo cruza. Ninguna de las dos existe: una probabilidad
vive **entre 0 y 1**, y a la recta eso nadie se lo dijo.

La **regresión logística** resuelve esto aplastando la recta dentro del rango 0–1 con una función
que le da forma de **S** (la *sigmoide*). Nunca toca el 0 ni el 1, solo se les acerca.

In [ ]:
logistica = smf.logit("es_us ~ salario", data=datos_log).fit(disp=0)

x_grilla = pd.DataFrame({"salario": np.linspace(0, 400_000, 300)})
x_grilla["prob"] = logistica.predict(x_grilla)

plt.figure(figsize=(11, 5))
plt.scatter(datos_log["salario"], datos_log["es_us"], alpha=0.25, color="#4C9BE8")
plt.plot(x_grilla["salario"], x_grilla["prob"], color="darkgreen", linewidth=3,
         label="Regresión LOGÍSTICA (la curva S)")

plt.axhline(1, color="gray", linestyle=":", linewidth=1)
plt.axhline(0, color="gray", linestyle=":", linewidth=1)
plt.title("La sigmoide se queda siempre entre 0 y 1", fontsize=14, fontweight="bold")
plt.xlabel("Sueldo anual (USD)")
plt.ylabel("Probabilidad de vivir en EEUU")
plt.legend()
plt.show()

In [ ]:
for sueldo in [20_000, 60_000, 120_000, 250_000]:
    p = logistica.predict(pd.DataFrame({"salario": [sueldo]}))[0]
    print(f"Sueldo ${sueldo:>7,}  →  probabilidad de vivir en EEUU: {p:5.1%}")

aciertos = ((logistica.predict(datos_log) > 0.5).astype(int) == datos_log["es_us"]).mean()
print()
print(f"Exactitud del modelo: {aciertos:.1%}")
print(f"(Si adivináramos siempre 'no vive en EEUU' acertaríamos {1 - datos_log['es_us'].mean():.1%})")

💡 **Esta sí funciona.** A diferencia del modelo de sueldos, acá el modelo acierta el 84,8% de las
veces, bastante por encima del 74,7% que sacarías adivinando siempre lo mismo.

Mismo archivo, misma variable de entrada, **dos preguntas distintas y dos máquinas distintas**:

| | Regresión **lineal** | Regresión **logística** |
|---|---|---|
| Qué predice | un **número** | una **probabilidad** de sí/no |
| Forma | una recta | una curva en S |
| Nuestro ejemplo | ¿cuánto gana? | ¿vive en EEUU? |
| Se evalúa con | **RMSE**, R² | exactitud, matriz de confusión |
| En Python | `smf.ols` / `LinearRegression` | `smf.logit` / `LogisticRegression` |

⚠️ Y no olvides la lección del bloque anterior: que el modelo acierte **no** significa que el
sueldo *cause* vivir en Estados Unidos. Significa que en el mundo real los sueldos de EEUU son
mucho más altos — que es un hecho incómodo, no una relación causal.

---
### 👨‍🏫 Evaluar una clasificación: la exactitud no alcanza

Dijimos que el modelo acierta el **84,8%**. Suena bien. Pero la exactitud sola engaña, y hay una
forma estándar de ver *cómo* acierta y *cómo* se equivoca: la **matriz de confusión**.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_clase = (logistica.predict(datos_log) > 0.5).astype(int)
mc = confusion_matrix(datos_log["es_us"], pred_clase)

print("                    PREDIJO no-US   PREDIJO US")
print(f"  ERA no-US  {mc[0, 0]:>14}  {mc[0, 1]:>12}")
print(f"  ERA US     {mc[1, 0]:>14}  {mc[1, 1]:>12}")

In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(mc, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predijo: no-US", "Predijo: US"],
            yticklabels=["Era: no-US", "Era: US"])
plt.title("Matriz de confusión", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

👨‍🏫 **Cómo se lee.** La diagonal son los aciertos; fuera de la diagonal, los errores. Pero hay
**dos tipos de error distintos**, y casi nunca cuestan lo mismo:

| Error | Qué pasó | En otro contexto |
|---|---|---|
| **Falso positivo** | dijo "US" y no lo era | dijiste "es fraude" y era un cliente legítimo |
| **Falso negativo** | dijo "no-US" y sí lo era | dijiste "no es fraude" y te robaron |

💡 **Un banco prefiere molestar a un cliente honesto antes que dejar pasar un fraude.** Un
diagnóstico médico, lo mismo. Por eso la exactitud sola no alcanza: **hay que saber en qué
dirección te equivocás.**

In [ ]:
print(classification_report(datos_log["es_us"], pred_clase,
                            target_names=["no-US", "US"], digits=3))

👨‍🏫 **Dos métricas más, que salen de esa tabla:**

- **Precisión** *(precision)*: de todos los que predije como US, ¿cuántos lo eran de verdad?
- **Sensibilidad** *(recall)*: de todos los que **eran** US, ¿a cuántos los encontré?

Casi siempre están en tensión: si querés encontrar a todos los US (recall alto), vas a marcar de
más y bajás la precisión. Cuál priorizás **es una decisión de negocio, no de estadística**.

### ⭐ Desafío opcional — el umbral es una decisión, no una ley

El modelo no devuelve "US / no-US": devuelve una **probabilidad**. Somos nosotros los que decidimos
que arriba de **0,5** se declare "US". Ese número no viene de ningún lado: es una convención.

Movelo y mirá qué pasa.

In [ ]:
probas = logistica.predict(datos_log)

print(f"{'Umbral':>8} {'Exactitud':>11} {'Marcados como US':>18}")
print("  " + "-" * 38)
for umbral in [0.20, 0.35, 0.50, 0.65, 0.80]:
    pred_u = (probas > umbral).astype(int)
    exactitud = (pred_u == datos_log["es_us"]).mean()
    print(f"{umbral:>8.2f} {exactitud:>11.1%} {pred_u.sum():>18}")

💡 **Mismo modelo, distinto umbral, distintos resultados.** No re-entrenaste nada: solo cambiaste
dónde cortás. Elegir ese punto de corte es una de las decisiones más importantes —y más ignoradas—
de un modelo de clasificación.

### ✍️ TU TURNO 5

Usá el modelo logístico para responder una pregunta concreta:

**¿Qué probabilidad tiene de vivir en EEUU alguien que gana $150.000 al año?**

💡 *Pista:* `logistica.predict(pd.DataFrame({"salario": [150_000]}))`.

In [ ]:
# TU TURNO 5: probabilidad de vivir en EEUU con un sueldo de $150.000
# Tu código aquí

### ✍️ TU TURNO 6 — el ejercicio integrador

Armá, de punta a punta, un modelo que prediga el **salario** a partir de **`exp_laboral`**, y
reportá las **tres** cosas que aprendiste hoy a reportar:

1. el coeficiente y su p-valor,
2. el **RMSE**,
3. el **R²**.

Y después escribí, en una línea de texto, si le creerías o no.

💡 *Pista:* ya hiciste cada paso por separado. Es juntarlos.

In [ ]:
# TU TURNO 6: modelo completo salario ~ exp_laboral, con las tres métricas
# Tu código aquí

---
# Cierre

## La respuesta completa

> **¿Cuánto va a ganar un programador con 5 años de experiencia?**

**La respuesta ingenua:** $80.853.

**La respuesta honesta:** *"No lo sé, y puedo demostrarte por qué no se puede saber con este dato.
Mi mejor estimación es $80.853, pero mi error típico es de $78.343 — un 86% del sueldo promedio.
La relación entre experiencia y sueldo es real (p ≈ 10⁻²⁴), pero explica apenas el 8% de lo que
hace que una persona gane lo que gana. Si querés predecir sueldos de verdad, necesitás saber el
país, la empresa y el rol."*

**Esa segunda respuesta es la que te va a hacer valioso en un trabajo.** Cualquiera corre un
`.fit()`. Muy pocos saben decir cuánto vale el resultado.

## Lo que dominás ahora

| Concepto | En Python | Cuándo lo usás |
|---|---|---|
| Regresión lineal simple | `smf.ols("y ~ x", data=df).fit()` | predecir un número |
| Intercepto y coeficiente | `.params` | interpretar el modelo |
| Significancia de la pendiente | `.pvalues` | ¿la relación es real? |
| R² | `.rsquared` / `.score()` | ¿cuánto explica? |
| MAE / MSE / **RMSE** | `sklearn.metrics` | **¿cuánto se equivoca?** |
| Residuos | `real - predicho` | ver **dónde** falla |
| Regresión múltiple | `"y ~ x1 + x2 + x3"` | varias variables a la vez |
| Colinealidad | `.corr()`, VIF | detectar predictores gemelos |
| Regresión logística | `smf.logit(...)` | predecir sí/no |
| Dialecto del TP | `LinearRegression()`, `reshape(-1,1)` | tu Assignment 3 |

## Las ideas que valen más que las funciones

1. **Estadísticamente significativo no quiere decir útil.** El p-valor dice si la relación existe;
   el RMSE dice si te sirve. Son preguntas distintas.
2. **El RMSE es el tamaño típico de tu error, en las unidades de lo que predecís.** Si no lo podés
   decir en pesos, dólares o minutos, no lo entendiste.
3. **Un R² bajo no es un error de cálculo: es información sobre el mundo.** A veces la respuesta
   honesta es "con estos datos no se puede".
4. **Nunca creas un número de regresión sin mirar el gráfico.** El cuarteto de Anscombe: cuatro
   realidades opuestas con el mismo R².
5. **El R² siempre sube al agregar variables**, aunque sean inútiles. No lo uses para decidir.
6. **Predictores que dicen lo mismo se anulan entre sí.** Agregar datos no siempre es agregar
   información.
7. **La herramienta que grita es tu amiga; la que calla, no.** Un nulo descartado en silencio te
   cambia el análisis sin avisarte.
8. **El modelo nunca sabe lo que no le contaste.** Los residuos grandes no son fallas: son la
   lista de todo lo que falta.

## ⚠️ Lo que NO podemos concluir hoy

- Que la experiencia **cause** un sueldo más alto. Encontramos una asociación, no una causa.
- Que estos números valgan para Bolivia: la encuesta es global y está dominada por EEUU y Europa.
- Que el modelo sirva para negociar tu sueldo. Con un error del 86%, no.

💡 Y esto es **exactamente** lo que arranca el 15 de agosto en el Módulo 3. Todo lo de hoy
—ajustar, predecir, medir el error— **es machine learning**. Lo único que cambia de acá en
adelante son modelos más flexibles que una recta. La pregunta *"¿cuánto se equivoca?"* va a
seguir siendo la misma.

## 📌 Tus tres tareas de esta unidad

| Qué | Dónde | Detalle |
|---|---|---|
| **Examen MD2 – Unidad 4** | Campus | Ya está abierto. Cierra el **22/08** |
| **Trabajo práctico MD2 – U4** | Campus | `student_scores.csv` (horas de estudio → nota). Es regresión lineal simple: **el bloque Rosetta de hoy tiene la sintaxis exacta** |
| **Foro 3** | Campus | Lineal vs. logística. Ya tenés el ejemplo: mismo archivo, dos preguntas, dos máquinas |

⚠️ **Aviso sobre el trabajo práctico:** el dataset tiene **25 filas** y un R² de **0,95**. Después
de lo de hoy ya sabés que un R² así en datos reales es sospechoso — ese archivo está armado para
que la recta salga linda. Resolvelo como te lo piden, pero mirá el gráfico y sacá tus conclusiones.

## Para seguir, si te quedaste con ganas

**En español:**
- **Seeing Theory** (Brown University) — [seeing-theory.brown.edu/regression-analysis/es.html](https://seeing-theory.brown.edu/regression-analysis/es.html)
  El cuarteto de Anscombe, pero **arrastrable**: movés los puntos y ves la recta cambiar.
- **PhET · Regresión de mínimos cuadrados** — [phet.colorado.edu](https://phet.colorado.edu/es/simulations/least-squares-regression)
  Ajustás tu propia recta a mano y comparás con la óptima. Media hora bien invertida.
- **DotCSV · Regresión Lineal y Mínimos Cuadrados** (YouTube) — la intuición matemática sin dolor.

**En inglés, si te animás:**
- **StatQuest** (Josh Starmer) — sus videos de *Linear Regression*, *R-squared* y *Logistic
  Regression* son el estándar mundial para principiantes.
- **Penn State STAT 501** — [online.stat.psu.edu/stat501](https://online.stat.psu.edu/stat501/)
  Curso universitario completo y gratuito. Lección 1 (la recta), 2 (R²), 12 (colinealidad).

**Documentación (para el TP):**
- [Fórmulas de statsmodels](https://www.statsmodels.org/stable/example_formulas.html) ·
  [LinearRegression de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

---

### Nos vemos

Hoy aprendiste a construir un modelo **y a desconfiar de él con argumentos**. Esas dos cosas
juntas son, literalmente, el trabajo.

*Módulo 2 · Clase 3 · Certificación en Inteligencia Artificial Generativa · UTEPSA / Khipus.ai*